# 🧹 Data Preprocessing Pipeline

This notebook demonstrates the end-to-end cleaning pipeline applied to all datasets.
Each step is shown with **before/after examples** so you can verify the quality.

### Pipeline Steps
1. Handle missing values
2. Remove duplicates
3. Clean text (HTML, URLs, mentions, special chars, lowercase)
4. Save cleaned datasets to `data/processed/`

---

## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import logging

from src.data.load_data import load_liar, load_isot, load_covid_tweets, load_twitter15_16
from src.data.preprocess import clean_text, preprocess_pipeline
from src.utils import setup_logging, get_data_dir

# Enable logging so we see pipeline progress
setup_logging(logging.INFO)

# Output directory for cleaned data
PROCESSED_DIR = get_data_dir('processed')
print(f'Processed data will be saved to: {PROCESSED_DIR}')
print('Setup complete ✓')

Processed data will be saved to: C:\Users\dawoo\OneDrive\Desktop\HASEEB\fake project\data\processed
Setup complete ✓


## Demonstrate Cleaning Steps

Before running on full datasets, let's see what each step does:

In [2]:
# Example texts to demonstrate cleaning
examples = [
    '<p>BREAKING: COVID vaccine causes <b>5G</b> signals!! Check http://fakenews.com/proof @DrFake #Plandemic</p>',
    'The unemployment rate has dropped to 3.5% according to @BLS_gov https://bls.gov/report #economy #jobs',
    '  RT @user123: This is    TOTALLY   fake!!!  🤡🤡🤡   ',
    'Scientists confirm: Earth is round. Published in Nature. DOI: 10.1038/s41586-021-03819-2',
    '',  # empty string
    None,  # None value
]

print('=== Before vs After Cleaning ===\n')
print(f'{"BEFORE":<65} | AFTER')
print('-' * 120)

for text in examples:
    cleaned = clean_text(str(text) if text else '')
    original = repr(text) if text is not None else 'None'
    print(f'{original:<65} | {repr(cleaned)}')

=== Before vs After Cleaning ===

BEFORE                                                            | AFTER
------------------------------------------------------------------------------------------------------------------------
'<p>BREAKING: COVID vaccine causes <b>5G</b> signals!! Check http://fakenews.com/proof @DrFake #Plandemic</p>' | 'breaking covid vaccine causes 5g signals check plandemic'
'The unemployment rate has dropped to 3.5% according to @BLS_gov https://bls.gov/report #economy #jobs' | 'the unemployment rate has dropped to 3 5 according to economy jobs'
'  RT @user123: This is    TOTALLY   fake!!!  🤡🤡🤡   '             | 'rt this is totally fake'
'Scientists confirm: Earth is round. Published in Nature. DOI: 10.1038/s41586-021-03819-2' | 'scientists confirm earth is round published in nature doi 10 1038 s41586 021 03819 2'
''                                                                | ''
None                                                              | ''


In [3]:
# Demonstrate with stopword removal + lemmatization
test_text = 'The scientists were running multiple experiments on the vaccines'

print(f'Original:                {test_text}')
print(f'Default clean:           {clean_text(test_text)}')
print(f'+ Stopword removal:      {clean_text(test_text, do_stopwords=True)}')
print(f'+ Lemmatization:         {clean_text(test_text, do_stopwords=True, do_lemmatize=True)}')

Original:                The scientists were running multiple experiments on the vaccines
Default clean:           the scientists were running multiple experiments on the vaccines


+ Stopword removal:      scientists running multiple experiments vaccines


+ Lemmatization:         scientist running multiple experiment vaccine


---
## 1. Preprocess LIAR Dataset

In [4]:
df_liar = load_liar()
print(f'Raw shape: {df_liar.shape}')
print(f'Missing text: {df_liar["text"].isna().sum()}')
print(f'Duplicates: {df_liar["text"].duplicated().sum()}')

00:22:39 | INFO     | src.data.load_data | Loading LIAR dataset from local TSV files...


00:22:39 | INFO     | src.data.load_data | LIAR dataset loaded: 12836 rows


Raw shape: (12836, 9)
Missing text: 0
Duplicates: 26


In [5]:
df_liar_clean = preprocess_pipeline(
    df_liar,
    text_col='text',
    label_col='label_binary',
    clean_kwargs={'do_stopwords': False, 'do_lemmatize': False}
)

print(f'\nCleaned shape: {df_liar_clean.shape}')
print(f'\n--- Sample (before → after) ---')
for i in range(min(5, len(df_liar_clean))):
    print(f'  BEFORE: {df_liar_clean.iloc[i]["text"][:80]}...')
    print(f'  AFTER:  {df_liar_clean.iloc[i]["text_clean"][:80]}...')
    print()

00:22:39 | INFO     | src.data.preprocess | Starting preprocessing (12836 rows)...


00:22:39 | INFO     | src.data.preprocess | Removed 26 duplicate rows (0.2%)


Cleaning text:   0%|          | 0/12810 [00:00<?, ?it/s]

Cleaning text:  10%|█         | 1330/12810 [00:00<00:00, 13189.04it/s]

Cleaning text:  23%|██▎       | 2948/12810 [00:00<00:00, 14926.05it/s]

...

Cleaning text:  78%|███████▊  | 9947/12810 [00:00<00:00, 16034.35it/s]

Cleaning text:  92%|█████████▏| 11823/12810 [00:00<00:00, 16878.44it/s]

Cleaning text: 100%|██████████| 12810/12810 [00:00<00:00, 16054.93it/s]

00:22:40 | INFO     | src.data.preprocess | Preprocessing complete: 12810 rows remaining



Cleaned shape: (12810, 10)

--- Sample (before → after) ---
  BEFORE: Says the Annies List political group supports third-trimester abortions on deman...
  AFTER:  says the annies list political group supports third trimester abortions on deman...

  BEFORE: When did the decline of coal start? It started when natural gas took off that st...
  AFTER:  when did the decline of coal start it started when natural gas took off that sta...

  BEFORE: Hillary Clinton agrees with John McCain "by voting to give George Bush the benef...
  AFTER:  hillary clinton agrees with john mccain by voting to give george bush the benefi...

  BEFORE: Health care reform legislation is likely to mandate free sex change surgeries....
  AFTER:  health care reform legislation is likely to mandate free sex change surgeries...

  BEFORE: The economic turnaround started at the end of my term....
  AFTER:  the economic turnaround started at the end of my term...



In [6]:
# Save
output_path = PROCESSED_DIR / 'liar_cleaned.csv'
df_liar_clean.to_csv(output_path, index=False)
print(f'✓ Saved to {output_path} ({len(df_liar_clean)} rows)')

✓ Saved to C:\Users\dawoo\OneDrive\Desktop\HASEEB\fake project\data\processed\liar_cleaned.csv (12810 rows)

---
## 2. Preprocess ISOT Dataset

> ⚠️ Requires manual download. Skip if not yet downloaded.

In [7]:
try:
    df_isot = load_isot()
    print(f'Raw shape: {df_isot.shape}')
    print(f'Missing text: {df_isot["text"].isna().sum()}')
    print(f'Duplicates: {df_isot["text"].duplicated().sum()}')

    df_isot_clean = preprocess_pipeline(df_isot, text_col='text', label_col='label_binary')

    output_path = PROCESSED_DIR / 'isot_cleaned.csv'
    df_isot_clean.to_csv(output_path, index=False)
    print(f'\n✓ Saved to {output_path} ({len(df_isot_clean)} rows)')

except FileNotFoundError as e:
    print(f'⚠️ Skipping ISOT: {e}')

00:22:40 | INFO     | src.data.load_data | Loading ISOT Fake News dataset...


00:22:42 | INFO     | src.data.load_data | ISOT dataset loaded: 44898 rows


Raw shape: (44898, 7)
Missing text: 0
Duplicates: 5795
00:22:43 | INFO     | src.data.preprocess | Starting preprocessing (44898 rows)...


00:22:43 | INFO     | src.data.preprocess | Removed 5795 duplicate rows (12.9%)


Cleaning text:   0%|          | 0/39103 [00:00<?, ?it/s]

...

Cleaning text:  98%|█████████▊| 38442/39103 [00:13<00:00, 3256.69it/s]

Cleaning text:  99%|█████████▉| 38830/39103 [00:13<00:00, 3419.12it/s]

Cleaning text: 100%|██████████| 39103/39103 [00:13<00:00, 2862.40it/s]

00:22:57 | INFO     | src.data.preprocess | Preprocessing complete: 39098 rows remaining



✓ Saved to C:\Users\dawoo\OneDrive\Desktop\HASEEB\fake project\data\processed\isot_cleaned.csv (39098 rows)


---
## 3. Preprocess COVID-19 Tweets

> ⚠️ Requires manual download. Skip if not yet downloaded.

In [8]:
try:
    df_covid = load_covid_tweets()
    print(f'Raw shape: {df_covid.shape}')

    # For tweets, we use the same pipeline but it's already tweet-friendly
    df_covid_clean = preprocess_pipeline(df_covid, text_col='text', label_col='label_binary')

    output_path = PROCESSED_DIR / 'covid_tweets_cleaned.csv'
    df_covid_clean.to_csv(output_path, index=False)
    print(f'\n✓ Saved to {output_path} ({len(df_covid_clean)} rows)')

except FileNotFoundError as e:
    print(f'⚠️ Skipping COVID tweets: {e}')

00:23:03 | INFO     | src.data.load_data |   Loaded Constraint_Train.csv: 6420 rows


00:23:03 | INFO     | src.data.load_data |   Loaded Constraint_Val.csv: 2140 rows


00:23:03 | INFO     | src.data.load_data |   Loaded Constraint_Test.csv: 2140 rows


00:23:03 | INFO     | src.data.load_data | COVID-19 tweets loaded: 10700 rows


Raw shape: (10700, 5)
00:23:03 | INFO     | src.data.preprocess | Starting preprocessing (10700 rows)...


...

Cleaning text:  79%|███████▊  | 8402/10699 [00:00<00:00, 10799.53it/s]

Cleaning text:  91%|█████████ | 9715/10699 [00:00<00:00, 11520.02it/s]

Cleaning text: 100%|██████████| 10699/10699 [00:01<00:00, 10647.32it/s]

00:23:04 | INFO     | src.data.preprocess | Preprocessing complete: 10699 rows remaining



✓ Saved to C:\Users\dawoo\OneDrive\Desktop\HASEEB\fake project\data\processed\covid_tweets_cleaned.csv (10699 rows)


---
## 4. Preprocess Twitter15/16

> ⚠️ Requires manual download. Skip if not yet downloaded.

In [9]:
try:
    df_twitter = load_twitter15_16()
    print(f'Raw shape: {df_twitter.shape}')

    # Filter out unverified for the binary task
    df_twitter_binary = df_twitter[df_twitter['label_binary'] != -1].copy()
    print(f'After removing unverified: {df_twitter_binary.shape}')

    df_twitter_clean = preprocess_pipeline(
        df_twitter_binary, text_col='text', label_col='label_binary'
    )

    output_path = PROCESSED_DIR / 'twitter15_16_cleaned.csv'
    df_twitter_clean.to_csv(output_path, index=False)
    print(f'\n✓ Saved to {output_path} ({len(df_twitter_clean)} rows)')

except FileNotFoundError as e:
    print(f'⚠️ Skipping Twitter15/16: {e}')

00:23:04 | INFO     | src.data.load_data |   Loaded twitter15 (Twitter15_label_All.txt): 1490 rows


00:23:04 | INFO     | src.data.load_data |   Loaded twitter16 (Twitter16_label_All.txt): 818 rows


00:23:04 | INFO     | src.data.load_data | Twitter15/16 loaded: 2308 rows


Raw shape: (2308, 13)
After removing unverified: (1733, 13)
00:23:04 | INFO     | src.data.preprocess | Starting preprocessing (1733 rows)...


00:23:04 | INFO     | src.data.preprocess | Removed 1426 duplicate rows (82.3%)


Cleaning text:   0%|          | 0/307 [00:00<?, ?it/s]

Cleaning text: 100%|██████████| 307/307 [00:00<00:00, 16838.21it/s]

00:23:04 | INFO     | src.data.preprocess | Preprocessing complete: 307 rows remaining



✓ Saved to C:\Users\dawoo\OneDrive\Desktop\HASEEB\fake project\data\processed\twitter15_16_cleaned.csv (307 rows)


---
## 5. Validation Checks

Verify all cleaned datasets pass quality checks.

In [10]:
import glob

print('=== Validation Report ===\n')

for csv_file in sorted(PROCESSED_DIR.glob('*.csv')):
    df = pd.read_csv(csv_file)
    name = csv_file.stem
    
    checks = {
        'Rows': len(df),
        'Null text_clean': df['text_clean'].isna().sum() if 'text_clean' in df.columns else 'N/A',
        'Empty text_clean': (df['text_clean'].str.len() == 0).sum() if 'text_clean' in df.columns else 'N/A',
        'Duplicate text': df['text_clean'].duplicated().sum() if 'text_clean' in df.columns else 'N/A',
        'Null labels': df['label_binary'].isna().sum() if 'label_binary' in df.columns else 'N/A',
        'Label dist': df['label_binary'].value_counts().to_dict() if 'label_binary' in df.columns else 'N/A',
    }
    
    status = '✓' if checks.get('Null text_clean', 1) == 0 and checks.get('Empty text_clean', 1) == 0 else '⚠️'
    print(f'{status} {name}')
    for k, v in checks.items():
        print(f'    {k}: {v}')
    print()

=== Validation Report ===



✓ covid_tweets_cleaned
    Rows: 10699
    Null text_clean: 0
    Empty text_clean: 0
    Duplicate text: 336
    Null labels: 0
    Label dist: {0: 6619, 1: 4080}



✓ isot_cleaned
    Rows: 39098
    Null text_clean: 0
    Empty text_clean: 0
    Duplicate text: 272
    Null labels: 0
    Label dist: {0: 21196, 1: 17902}



✓ liar_cleaned
    Rows: 12810
    Null text_clean: 0
    Empty text_clean: 0
    Duplicate text: 7
    Null labels: 0
    Label dist: {0: 7156, 1: 5654}

✓ twitter15_16_cleaned
    Rows: 307
    Null text_clean: 0
    Empty text_clean: 0
    Duplicate text: 0
    Null labels: 0
    Label dist: {1: 238, 0: 69}



---

## ✅ Preprocessing Complete

All cleaned datasets are saved in `data/processed/`. They are ready for:
- Feature engineering (TF-IDF, embeddings)
- Model training (Logistic Regression, BERT, etc.)

### Files Created
| File | Description |
|------|-------------|
| `liar_cleaned.csv` | LIAR dataset — cleaned political statements |
| `isot_cleaned.csv` | ISOT dataset — cleaned news articles |
| `covid_tweets_cleaned.csv` | COVID-19 fake tweets — cleaned |
| `twitter15_16_cleaned.csv` | Twitter rumor tweets — cleaned |